# Lab F6. Breaks and nonlinearity

A model estimated on the past forecasts the future only if the future behaves like the past. This lab looks at
two ways it may not: a break, after which the old parameters are wrong, and a nonlinearity, where the dynamics
depend on the state of the economy.

**Help:** for any pymetrix function, type its name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call. The last cell lists every pymetrix function, class and method this lab uses.

Topics: tests for a break; rolling and expanding windows, and averaging over windows; threshold autoregressions;
Hansen's and Teräsvirta's tests of linearity; forecasting from a nonlinear model (Elliott and Timmermann,
chapters 8 and 19).

In [ ]:
import numpy as np
import pandas as pd

from pymetrix import Dataset, Equation, ThresholdAR, backtest, combine_forecasts, terasvirta

quarters = pd.period_range("1965Q1", periods=240, freq="Q")

## 1. A break in the mean

Growth follows $y_t = \mu_t (1 - 0.5) + 0.5\,y_{t-1} + \varepsilon_t$, with mean growth $\mu_t = 3$ until 2004Q4
and $1$ from 2005Q1: a slowdown that a model estimated on the whole past will be slow to accept.

In [ ]:
rng = np.random.default_rng(19)
mu = np.where(quarters >= pd.Period("2005Q1", freq="Q"), 1.0, 3.0)
y = np.zeros(len(quarters))
y[0] = 3.0
for t in range(1, len(quarters)):
    y[t] = mu[t] * 0.5 + 0.5 * y[t - 1] + rng.standard_normal()
growth = Dataset.from_frame(pd.DataFrame({"y": y}, index=quarters), source="simulated, break in the mean in 2005Q1")

ar = Equation("y ~ y[-1]").fit(growth)
{"Chow, at 2005Q1": round(float(ar.test("chow", at="2005Q1").p_value), 4), "CUSUM": round(float(ar.test("cusum").p_value), 4)}

Three ways to forecast through the break, one quarter ahead from 1995Q1 to 2024Q3: all the data to each origin
(expanding), the last 24 quarters (rolling), and the average of the two (a simple way to hedge across windows,
as Pesaran and Timmermann suggest).

In [ ]:
origins = ("1995Q1", "2024Q3")
expanding = backtest(ar, growth, origins, horizon=1, label="expanding")
rolling = backtest(ar, growth, origins, horizon=1, window=24, label="rolling, 24 quarters")
average = combine_forecasts([expanding, rolling], method="equal", label="average of the windows")


def rmse(bt, start, end):
    e = bt.errors.set_index("target")["error"]
    return float(np.sqrt((e[(e.index >= pd.Period(start, freq="Q")) & (e.index <= pd.Period(end, freq="Q"))] ** 2).mean()))


periods = {"before the break (1995-2004)": ("1995Q2", "2004Q4"), "first five years after (2005-2009)": ("2005Q1", "2009Q4"),
           "long after (2010-2024)": ("2010Q1", "2024Q4")}
pd.DataFrame({name: {label: rmse(bt, *span) for label, span in periods.items()} for name, bt in
              (("expanding", expanding), ("rolling", rolling), ("average", average))})

**Check it yourself.** The rolling forecast from 2010Q1 is an AR(1) fitted by least squares to the 24 quarters
2004Q2-2010Q1, then applied to $y$ in 2010Q1.

In [ ]:
window = growth.frame["y"].loc[pd.Period("2004Q1", freq="Q"): pd.Period("2010Q1", freq="Q")].to_numpy()
X = np.column_stack([np.ones(24), window[:-1]])
b = np.linalg.lstsq(X, window[1:], rcond=None)[0]
row = rolling.errors.query("origin == '2010Q1'").iloc[0]
assert np.isclose(b[0] + b[1] * window[-1], row["forecast"])
print(f"rolling forecast for 2010Q2 by hand {b[0] + b[1] * window[-1]:.6f}; pymetrix {row['forecast']:.6f}")

**Questions.**

1. Which window wins before the break, just after it, and long after? Why does the expanding window recover so
   slowly?
2. Try windows of 12 and 48 quarters. What is the trade-off?
3. The break date here is known. Suppose it were not: what would the CUSUM test have told a forecaster in 2006
   (fit on `span=(None, "2006Q4")`)?

## 2. A threshold autoregression

Recessions are short and expansions long: when growth is negative it rebounds quickly; when it is positive it
persists. A two-regime threshold AR captures that:

$y_t = 0.6 + 0.2\,y_{t-1} + \varepsilon_t$ when $y_{t-1} \le 0$, and $y_t = 0.2 + 0.8\,y_{t-1} + \varepsilon_t$
when $y_{t-1} > 0$.

In [ ]:
rng = np.random.default_rng(8)
z = np.zeros(len(quarters))
for t in range(1, len(quarters)):
    z[t] = (0.6 + 0.2 * z[t - 1] if z[t - 1] <= 0 else 0.2 + 0.8 * z[t - 1]) + rng.standard_normal()
cycle = Dataset.from_frame(pd.DataFrame({"y": z}, index=quarters), source="simulated threshold AR")
setar = ThresholdAR("y", lags=1).fit(cycle)
setar.summary()

**Check it yourself.** The threshold is found by trying every candidate (each observed $y_{t-1}$, keeping at
least 15% of the observations in each regime), fitting least squares in each regime, and keeping the one with the
smallest total sum of squared residuals.

In [ ]:
lagged, current = z[:-1], z[1:]
candidates = np.sort(lagged)[int(np.ceil(0.15 * len(lagged))) - 1: len(lagged) - int(np.ceil(0.15 * len(lagged)))]


def ssr(mask):
    X = np.column_stack([np.ones(mask.sum()), lagged[mask]])
    e = current[mask] - X @ np.linalg.lstsq(X, current[mask], rcond=None)[0]
    return e @ e


total = [ssr(lagged <= c) + ssr(lagged > c) for c in candidates]
threshold = candidates[int(np.argmin(total))]
assert np.isclose(threshold, setar.threshold) and np.isclose(min(total), setar.ssr)
print(f"threshold by grid search {threshold:.4f}; pymetrix {setar.threshold:.4f}")

Is the nonlinearity real? Under the null of a linear AR the threshold is not identified, so the usual F test does
not apply: Hansen's test bootstraps the distribution of the largest F over all thresholds. Teräsvirta's LM test
asks the same question against a smooth transition, and needs no bootstrap.

In [ ]:
hansen = setar.linearity_test(reps=300)
lm = terasvirta(cycle, "y", lags=1, delay=1)
pd.DataFrame({"statistic": [hansen.statistic, lm.statistic], "p-value": [hansen.p_value, lm.p_value]},
             index=["Hansen sup-F (bootstrap)", "Teräsvirta LM"])

## 3. Does the nonlinearity help forecast?

A significant nonlinearity in sample need not forecast better out of sample. Forecasts from the threshold AR
beyond one step are simulated (the regime two quarters ahead depends on the shock next quarter). Compare it with
the linear AR(1), one and four quarters ahead, from 2000Q1 to 2023Q4.

In [ ]:
bt_setar = backtest(setar, cycle, ("2000Q1", "2023Q4"), horizon=4, label="threshold AR")
bt_ar = backtest(Equation("y ~ y[-1]").fit(cycle), cycle, ("2000Q1", "2023Q4"), horizon=4, label="AR(1)")
pd.DataFrame({"threshold AR": bt_setar.scores(["y"])["RMSE"].to_numpy(), "AR(1)": bt_ar.scores(["y"])["RMSE"].to_numpy()},
             index=pd.Index(range(1, 5), name="horizon"))

In [ ]:
{f"h={h}": round(float(bt_setar.compare(bt_ar, "y", h).p_value), 4) for h in (1, 4)}

**Questions.**

4. Where do the two models' forecasts differ most: after negative quarters or positive ones? (Group
   `bt_setar.errors` and `bt_ar.errors` by the sign of $y$ at the origin.)
5. The threshold AR's density forecasts come from its simulations: compare the two models' log scores
   (`bt_setar.density_scores()`; the AR(1) has densities one step ahead).

In the app: **Estimate → Nonlinear → Threshold autoregression** (its Tests tab has both linearity tests); **Test → Run test**
has the Chow and CUSUM tests; **Forecast → Backtest** has the rolling window on its Advanced tab.

## pymetrix used in this lab

Every pymetrix function, class and method this lab calls, with its arguments and the first line of its documentation. For the full documentation, type the name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call.

### From `pymetrix`

| | |
|---|---|
| `backtest(fitted, data, origins, horizon, vintages, variable, label, window)` | Re-estimate at each origin from `origins[0]` to `origins[1]` and forecast `horizon` periods ahead. |
| `combine_forecasts(backtests, method, min_history, label)` | Combine the forecasts of several backtests into one, and evaluate it as a backtest of its own. |
| `Equation(text, params, family, priors, name, errors, variance)` | An equation written as text: `"c ~ y + c[-1]"` to estimate, or `"y = c + i + g"` (an identity). |
| `terasvirta(data, series, lags, delay, span)` | Teräsvirta's (1994) LM test of linearity against a smooth-transition AR: regress the AR(lags) residuals on the lags, and the lags times y_(t-d), y_(t-d)^2 and y_(t-d)^3 (a third-order expansion of the transition); F test of the added terms. |
| `ThresholdAR(series, lags, delay, trim)` | A two-regime self-exciting threshold autoregression (SETAR), by conditional least squares over the threshold and the delay. |

### Methods and properties of the results

| | |
|---|---|
| `Backtest.compare(other, variable, horizon, loss, test, tau, a)` | Test whether these forecasts and `other`'s are equally accurate at one horizon, on the targets both forecast. |
| `Backtest.scores(variables)` | n, RMSE, MAE and mean error by variable and horizon (a two-level index). |
| `Dataset.from_frame(df, time, entity, freq, variables, meta, source)` | Build a Dataset from a DataFrame, naming its time and entity columns. |
| `Equation.fit(data, span, where, method, weights, missing)` | Estimate the equation on `data`: by least squares unless `method` says otherwise (`OLS(cov="hac")`, `IV(...)`, `ML()`, `Ridge()`, ...). |
| `FittedEquation.test(name, *args, **opts)` | Run a registered test on this fit by name, e.g. `fit.test("breusch_godfrey", lags=4)` or `fit.test("chow", at="2008Q1")`; `tests()` lists the ones that apply. |
| `FittedThresholdAR.linearity_test(reps, seed)` | Hansen's (1996, 1997) test of the linear AR against the threshold model: sup-F over the thresholds (and delays, when chosen), F = n (SSR_linear - SSR_threshold) / SSR_threshold, with a fixed-regressor bootstrap p-value (the threshold is not identified under the null, so the usual F distribution does not apply). |
| `FittedThresholdAR.summary()` | The coefficients and standard errors in each regime, with the threshold, the delay and the fit. |
| `ThresholdAR.fit(data, span)` | Estimate on the periods of `span` (first, last): least squares in each regime at every candidate threshold (and delay, when it is 0), keeping the one with the smallest sum of squared residuals. |